# Example 1: MMS on the Unit Square

## Problem Setup

We verify the DG advection-diffusion implementation using the **Method of Manufactured Solutions (MMS)** on a unit square domain $\Omega = [0,1]^2$ with a prescribed uniform upward velocity field $\mathbf{w} = (0, w_\text{mag})$.

The governing equation is:

$$
\nabla \cdot (\mathbf{w} u) - \nabla \cdot (D \nabla u) = f \quad \text{in } \Omega
$$

## Boundary Conditions

All four sides carry a **Dirichlet condition** for the diffusion term ($\Gamma_D = \partial\Omega$), imposed weakly via SIPG. The advection inflow/outflow split is determined by $\mathbf{w}\cdot\mathbf{n}$:

| Boundary | Location | $\mathbf{w}\cdot\mathbf{n}$ | Advection type |
|---|---|---|---|
| Bottom | $y = 0$ | $-w_\text{mag} < 0$ | Inflow — $u = u_\text{exact}$ prescribed |
| Top | $y = 1$ | $+w_\text{mag} > 0$ | Outflow — interior trace used |
| Left / Right | $x = 0,\, 1$ | $0$ | Wall — no advective flux |

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

matplotlib.rcParams['svg.fonttype'] = 'none'

fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.add_patch(patches.Rectangle((0, 0), 1, 1, facecolor='white', edgecolor='black', linewidth=1.5, zorder=1))
ax.text(0.5, 0.5, r'$\Omega$', ha='center', va='center', fontsize=18)
ax.text(0.5, -0.06, r'$\Gamma_D$ (inflow)',  ha='center', va='top',    fontsize=10)
ax.text(0.5,  1.06, r'$\Gamma_D$ (outflow)', ha='center', va='bottom', fontsize=10)
ax.text(-0.06, 0.5, r'$\Gamma_D$ (wall)',    ha='right',  va='center', fontsize=10)
ax.text( 1.06, 0.5, r'$\Gamma_D$ (wall)',    ha='left',   va='center', fontsize=10)
for xi in [0.3, 0.7]:
    for yi in [0.2, 0.5, 0.78]:
        ax.annotate('', xy=(xi, yi + 0.11), xytext=(xi, yi),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.2), zorder=2)
ax.text(0.82, 0.88, r'$\mathbf{w}$', fontsize=13, ha='center')
ax.set_xlim(-0.35, 1.35); ax.set_ylim(-0.18, 1.18)
ax.set_aspect('equal'); ax.axis('off')
plt.tight_layout()
fig.savefig('../images/domain_mwe1.svg', bbox_inches='tight')
plt.close(fig)

:::{image} ../images/domain_mwe1.svg
:align: center
:width: 55%
:::

## Manufactured Solution

The exact solution is chosen to contain both a smooth oscillatory component and an exponential boundary layer near the outflow boundary ($y = 1$):

$$
u_\text{exact}(x, y) = \cos(\pi x) \cdot \frac{1 - e^{(y-1)/D}}{1 - e^{-2/D}} + \frac{1}{2}\cos(\pi x)\sin(\pi y)
$$

The first term produces a layer of thickness $\mathcal{O}(D)$ near $y=1$; for small $D$ (large Péclet number) this layer becomes sharp. The source term $f$ is derived analytically from $u_\text{exact}$ so that it satisfies the PDE exactly.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['svg.fonttype'] = 'none'

D = 0.1

def u_exact(x, y, D):
    return (np.cos(np.pi * x) * (1 - np.exp((y - 1) / D)) / (1 - np.exp(-2 / D))
            + 0.5 * np.cos(np.pi * x) * np.sin(np.pi * y))

x = np.linspace(0, 1, 300)
y = np.linspace(0, 1, 300)
X, Y = np.meshgrid(x, y)
U = u_exact(X, Y, D)

fig, ax = plt.subplots(figsize=(4.5, 4))
cf = ax.pcolormesh(X, Y, U, cmap='RdBu_r', shading='auto',
                   edgecolors='face', rasterized=True)
cbar = plt.colorbar(cf, ax=ax, shrink=0.9)
cbar.set_label(r'$u_\mathrm{exact}$', fontsize=10)
ax.set_aspect('equal')
ax.set_xlabel(r'$x$', fontsize=11)
ax.set_ylabel(r'$y$', fontsize=11)
ax.set_title(r'Manufactured solution ($D = 0.1$)', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig('../images/solution_mwe1.svg', bbox_inches='tight', dpi=200)
plt.close(fig)

:::{image} ../images/solution_mwe1.svg
:align: center
:width: 65%
:::

## Convergence Test

The mesh is refined uniformly ($N = 8, 16, \ldots, 256$) and the $L^2$ error $\|u_h - u_\text{exact}\|_{L^2(\Omega)}$ is tracked. For a degree-$p$ DG space the expected optimal convergence rate is $\mathcal{O}(h^{p+1})$.

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc

import matplotlib.pyplot as plt
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.io import VTXWriter


def exact_solution(x, D):
    return ufl.cos(ufl.pi * x[0]) * (1 - ufl.exp((x[1] - 1) / D)) / (
        1 - ufl.exp(-2 / D)
    ) + 0.5 * ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])


def solve_advection_diffusion(N, degree=1, D_value=0.1, w_mag=1.0):
    msh = mesh.create_unit_square(
        MPI.COMM_WORLD,
        N,
        N,
        cell_type=mesh.CellType.triangle,
        diagonal=mesh.DiagonalType.crossed,
    )

    print(type(msh))
    exit()

    V = fem.functionspace(msh, ("DG", degree))

    u = fem.Function(V)
    v = ufl.TestFunction(V)

    x = ufl.SpatialCoordinate(msh)
    n = ufl.FacetNormal(msh)
    h = ufl.CellDiameter(msh)

    D = fem.Constant(msh, PETSc.ScalarType(D_value))
    w = fem.Constant(msh, PETSc.ScalarType((0, w_mag)))

    u_exact = exact_solution(x, D)
    f = -ufl.div(D * ufl.grad(u_exact)) + ufl.dot(w, ufl.grad(u_exact))

    penalty = fem.Constant(msh, PETSc.ScalarType(10.0 * degree**2))

    dx = ufl.dx
    ds = ufl.ds
    dS = ufl.dS

    # Outflow indicator
    lmbda = ufl.conditional(ufl.gt(ufl.dot(w, n), 0), 1, 0)

    F = 0

    # Advection with upwind flux
    F += -ufl.inner(w * u, ufl.grad(v)) * dx
    F += ufl.inner(2 * ufl.avg(lmbda * w * u), ufl.jump(v, n)) * dS
    F += ufl.inner(lmbda * ufl.dot(w, n) * u, v) * ds

    # Diffusion, symmetric interior penalty
    F += D * ufl.inner(ufl.grad(u), ufl.grad(v)) * dx
    F += -D * ufl.inner(ufl.avg(ufl.grad(u)), ufl.jump(v, n)) * dS
    F += -D * ufl.inner(ufl.jump(u, n), ufl.avg(ufl.grad(v))) * dS
    F += D * (penalty / ufl.avg(h)) * ufl.inner(ufl.jump(u, n), ufl.jump(v, n)) * dS

    # Weak Dirichlet condition for diffusion on the whole boundary
    F += D * (
        -ufl.inner(ufl.grad(u), v * n) * ds
        - ufl.inner(ufl.grad(v), (u - u_exact) * n) * ds
        + (penalty / h) * ufl.inner(u - u_exact, v) * ds
    )

    # Inflow boundary condition for advection
    F += -ufl.inner((1 - lmbda) * ufl.dot(w, n) * u_exact, v) * ds

    # Source
    F += -ufl.inner(f, v) * dx

    J = ufl.derivative(F, u)

    problem = NonlinearProblem(
        F,
        u,
        J=J,
        petsc_options_prefix="advecdiff",
        petsc_options={
            "snes_type": "newtonls",
            "snes_linesearch_type": "none",
            "snes_rtol": 1e-10,
            "snes_atol": 1e-10,
            "snes_max_it": 20,
            "ksp_type": "preonly",
            "pc_type": "lu",
        },
    )

    u = problem.solve()

    u.x.scatter_forward()

    # writer = VTXWriter(msh.comm, "DG_solution.bp", u, "BP5")
    # writer.write(t=0)

    error_form = fem.form((u - u_exact) ** 2 * dx)
    local_error = fem.assemble_scalar(error_form)
    l2_error = np.sqrt(msh.comm.allreduce(local_error, op=MPI.SUM))

    return l2_error


def convergence_test(degree=1):
    Ns = [8, 16, 32, 64, 128, 256]
    # Ns = [64]
    errors = []

    for N in Ns:
        error = solve_advection_diffusion(N, degree=degree, w_mag=50.0)
        errors.append(error)
        if MPI.COMM_WORLD.rank == 0:
            print(f"N = {N:3d}, L2 error = {error:.6e}")

    if MPI.COMM_WORLD.rank == 0:
        h = np.array([1 / N for N in Ns], dtype=float)
        errors = np.array(errors)

        plt.figure()
        plt.plot(h, errors)

        plt.ylabel("L2 error")
        plt.xlabel("Element size (h)")
        plt.xscale("log")
        plt.yscale("log")
        plt.grid(True, which="major", ls="--", lw=0.5)

        ax = plt.gca()
        ax.loglog(h, 2 * h**1.5, linestyle="--", color="black")
        ax.annotate(
            "Order 1.5",
            (h[0], 2 * h[0] ** 1.5),
            textcoords="offset points",
            xytext=(10, 0),
        )
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()
        plt.show()


if __name__ == "__main__":
    convergence_test(degree=1)
